In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import kagglehub         # Download dataset from Kaggle
import pandas as pd
import shutil
import email
from email import policy
import re     # Regular expressions for text cleaning
from dateutil import parser as dateparser  # Flexible date string parsing

### Download dataset

In [42]:
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
df = pd.read_csv(f"{path}/emails.csv")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'enron-email-dataset' dataset.
Path to dataset files: /kaggle/input/enron-email-dataset


### Basic info check

In [43]:
# Check basic info
print(df.shape)
print("\n--- Column Names ---")
print(df.columns.tolist())
print("\n--- Data Types ---")
print(df.dtypes)
print("\n--- First few rows ---")
df.head()

(517401, 2)

--- Column Names ---
['file', 'message']

--- Data Types ---
file       object
message    object
dtype: object

--- First few rows ---


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


### Sample email

In [44]:
print(df['message'][0])

Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


### Missing values & duplicate check

In [45]:
print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate check ---")
print("Duplicates:", df.duplicated().sum())


--- Missing Values ---
file       0
message    0
dtype: int64

--- Duplicate check ---
Duplicates: 0


### Only Parse emails that mention 1997 - 2001 in the raw message

In [46]:
def parse_email(raw):
    msg = email.message_from_string(raw)
    body_part = msg.get_payload(decode=False)  # body extraction
    return {
        'message_id': msg.get('Message-ID', '').strip(),
        'date':       msg.get('Date', '').strip(),
        'sender':     msg.get('From', '').strip(),
        'recipients': msg.get('To', '').strip(),
        'cc':         msg.get('X-cc', '').strip(),
        'bcc':        msg.get('X-bcc', '').strip(),
        'in_reply_to': msg.get('In-Reply-To', '').strip(),
        'references':  msg.get('References', '').strip(),
        'subject':    msg.get('Subject', '').strip(),
        'body':       body_part if isinstance(body_part, str) else ''
    }

# Filter first
mask = df['message'].str.contains('199[7-9]|200[0-1]', regex=True, na=False)
df_relevant = df[mask]
print(f"Reduced from {len(df)} to {len(df_relevant)} emails")

# Then parse only filtered rows
parsed = df_relevant['message'].apply(parse_email)
df_parsed = pd.DataFrame(parsed.tolist())
print(f"Parsed: {len(df_parsed)} emails")

Reduced from 517401 to 485543 emails
Parsed: 485543 emails


### Standardize dates

In [47]:
def safe_parse_date(date_str):
    try:
        return dateparser.parse(date_str, fuzzy=True)
    except:
        return None

df_parsed['date'] = df_parsed['date'].apply(safe_parse_date)
df_parsed['date'] = pd.to_datetime(df_parsed['date'], utc=True, errors='coerce')
print("Date conversion done")

Date conversion done


### Deduplicate by message_id

In [48]:
before = len(df_parsed)
df_parsed = df_parsed.drop_duplicates(subset='message_id')
print(f"Removed {before - len(df_parsed)} duplicates")

Removed 0 duplicates


### Filter to 2000–2001

In [49]:
df_parsed = df_parsed.dropna(subset=['date'])
df_filtered = df_parsed[
    (df_parsed['date'].dt.year >= 1997) &
    (df_parsed['date'].dt.year <= 2001)
].copy()
print(f"Emails in 1997-2001: {len(df_filtered)}")

Emails in 1997-2001: 480627


### Clean body

In [50]:
def clean_body(series):
    # Work on the whole column at once instead of row by row
    series = series.fillna('')
    series = series.astype(str)

    # Remove reply chains (lines starting with >)
    series = series.str.replace(r'^>.*$', '', regex=True, flags=re.MULTILINE)
    # Remove "Original Message" blocks
    series = series.str.replace(r'-{3,}.*?original message.*', '', regex=True, flags=re.IGNORECASE)
    # Remove signature patterns
    series = series.str.replace(r'\n{2,}[-_]{2,}.*', '', regex=True)
    # Normalize whitespace
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()

    return series

df_filtered.loc[:, 'body_clean'] = clean_body(df_filtered['body'])

### Drop empty bodies + normalize text

In [51]:
# Drop emails with empty body
df_filtered = df_filtered[df_filtered['body_clean'].str.strip() != '']

# Lowercase normalization
df_filtered.loc[:, 'body_normalized'] = df_filtered['body_clean'].str.lower()

print(f"After removing empty bodies: {len(df_filtered)}")

After removing empty bodies: 480169


### Normalize sender email addresses

In [52]:
def extract_email(sender_str):
    """Extract clean email address from various formats:
    - 'phillip.allen@enron.com'  → 'phillip.allen@enron.com'
    - 'Phillip Allen <phillip.allen@enron.com>'  → 'phillip.allen@enron.com'
    - 'phillip.allen@enron.com (Phillip Allen)'  → 'phillip.allen@enron.com'
    """
    if not sender_str:
        return ''
    # Try to find email in angle brackets first
    match = re.search(r'<([^>]+)>', sender_str)
    if match:
        return match.group(1).strip().lower()
    # Try to find bare email address
    match = re.search(r'[\w\.\-\+]+@[\w\.\-]+', sender_str)
    if match:
        return match.group(0).strip().lower()
    # Fallback: return original lowercased
    return sender_str.strip().lower()

df_filtered.loc[:, 'sender_email'] = df_filtered['sender'].apply(extract_email)
df_filtered.loc[:, 'sender_domain'] = df_filtered['sender_email'].apply(
    lambda x: x.split('@')[-1] if '@' in x else ''
)

print(f"Unique normalized senders: {df_filtered['sender_email'].nunique()}")
print(f"Unique domains: {df_filtered['sender_domain'].nunique()}")
print(df_filtered[['sender', 'sender_email', 'sender_domain']].head(10))

Unique normalized senders: 17248
Unique domains: 4367
                    sender             sender_email sender_domain
0  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
1  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
2  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
3  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
4  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
5  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
6  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
7  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
8  phillip.allen@enron.com  phillip.allen@enron.com     enron.com
9  phillip.allen@enron.com  phillip.allen@enron.com     enron.com


### Keyword filter

In [53]:
WRONGDOING_KEYWORDS = [
    # SPE / off-balance-sheet
    'spe', 'special purpose', 'raptor', 'ljm', 'off-balance',
    # Mark-to-market
    'mark to market', 'mtm', 'fair value',
    # Concealment language
    'reserve', 'restate', 'write-down', 'write down',
    # Timing / quarter-end pressure
    'end of quarter', 'quarter end', 'close the books',
    # Key people
    'fastow', 'skilling', 'lay',
    # Audit
    'andersen', 'audit', 'disclosure'
]

pattern = '|'.join(WRONGDOING_KEYWORDS)
df_filtered.loc[:, 'has_wrongdoing_keyword'] = (
    df_filtered['body_normalized']
    .str.contains(pattern, regex=True, na=False)
)

flagged = df_filtered['has_wrongdoing_keyword'].sum()
print(f"Emails with at least one wrongdoing keyword: {flagged}")

# enron_flagged.csv as a file with Only emails with at least one wrongdoing keyword
df_flagged = df_filtered[df_filtered['has_wrongdoing_keyword']]
df_flagged.to_csv('/content/drive/MyDrive/INTA6450_Group/enron_flagged.csv', index=False)
print(f"Saved {len(df_flagged)} flagged emails to enron_flagged.csv")

Emails with at least one wrongdoing keyword: 146196
Saved 146196 flagged emails to enron_flagged.csv


### Summary printout

In [54]:
print(f"Total emails after cleaning: {len(df_filtered)}")
print(f"Date range: {df_filtered['date'].min()} to {df_filtered['date'].max()}")
print(f"Unique senders: {df_filtered['sender'].nunique()}")
print(df_filtered[['message_id','date','sender','recipients','subject','body_clean']].head())

Total emails after cleaning: 480169
Date range: 1997-01-01 17:22:03+00:00 to 2001-12-31 23:52:49+00:00
Unique senders: 17262
                                      message_id                      date  \
0  <18782981.1075855378110.JavaMail.evans@thyme> 2001-05-14 23:39:00+00:00   
1  <15464986.1075855378456.JavaMail.evans@thyme> 2001-05-04 20:51:00+00:00   
2  <24216240.1075855687451.JavaMail.evans@thyme> 2000-10-18 10:00:00+00:00   
3  <13505866.1075863688222.JavaMail.evans@thyme> 2000-10-23 13:13:00+00:00   
4  <30922949.1075863688243.JavaMail.evans@thyme> 2000-08-31 12:07:00+00:00   

                    sender               recipients    subject  \
0  phillip.allen@enron.com     tim.belden@enron.com              
1  phillip.allen@enron.com  john.lavorato@enron.com        Re:   
2  phillip.allen@enron.com   leah.arsdall@enron.com   Re: test   
3  phillip.allen@enron.com    randall.gay@enron.com              
4  phillip.allen@enron.com     greg.piper@enron.com  Re: Hello   

         

### Save cleaned version to shared Drive folder

In [55]:
cols_to_save = [
    'message_id',
    'date',
    'sender',
    'sender_email',
    'sender_domain',
    'recipients',
    'cc',
    'bcc',
    'in_reply_to',
    'references',
    'subject',
    'body_clean',          # original case, for NER/BERT
    'body_normalized',     # lowercase, for keyword matching
    'has_wrongdoing_keyword'
]

df_filtered[cols_to_save].to_csv('/content/drive/MyDrive/INTA6450_Group/enron_cleaned.csv', index=False)
print(f"Saved {len(df_filtered)} emails to shared Drive")
print(f"Columns saved: {cols_to_save}")

Saved 480169 emails to shared Drive
Columns saved: ['message_id', 'date', 'sender', 'sender_email', 'sender_domain', 'recipients', 'cc', 'bcc', 'in_reply_to', 'references', 'subject', 'body_clean', 'body_normalized', 'has_wrongdoing_keyword']
